# Wave Data Plots — Marconi Beach, Cape Cod

**Author:** Chelsea Volpano  
**Email:** cvolpano@contractor.usgs.gov | cvolpano@gmail.com  
**Date:** 2026-06-24  
**Created using Claude Sonnet 4.6**

---

Generates three figures from the downloaded wave CSVs:
1. **Time series** — Hs, Tp, and wave power with 7-day rolling mean
2. **Monthly box plots** — seasonal distribution of all three variables
3. **Hs–Tp scatter** — coloured by wave power with iso-power contours

Data sources:
- **WIS ST63064** — Oct–Dec 2024 (hindcast model)
- **NDBC 44020** — Oct 2024–Mar 2025 (observed)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec
from pathlib import Path

%matplotlib inline

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
OUT_DIR  = Path("wave_data")
PLOT_DIR = Path("wave_plots")
PLOT_DIR.mkdir(exist_ok=True)

WIS_FILE  = OUT_DIR / "WIS_ST63064_2024-10_to_2024-12.csv"
NDBC_FILE = OUT_DIR / "NDBC_44020_2024-10_to_2025-03.csv"

START = "2024-10-01"
END   = "2025-03-31"

# Wave power:  P = (rho * g^2) / (64 * pi) * Hs^2 * Tp  [kW/m]
RHO     = 1025.0
G       = 9.81
P_CONST = (RHO * G**2) / (64 * np.pi) / 1000

PALETTE = {
    "ndbc": "#2176AE",
    "wis":  "#E05C2A",
}

## 1. Load Data

In [ ]:
# NDBC fill values by column
NDBC_FILL = {"WVHT": 99.0, "DPD": 99.0, "APD": 99.0, "MWD": 999.0}

def load_ndbc(path):
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    df.index.name = "datetime"
    df = df[["WVHT", "DPD"]].rename(columns={"WVHT": "Hs", "DPD": "Tp"})
    df = df.apply(pd.to_numeric, errors="coerce")
    # Mask any remaining fill values
    df["Hs"] = df["Hs"].where(df["Hs"] < 99.0)
    df["Tp"] = df["Tp"].where(df["Tp"] < 99.0)
    df["P_kW_m"] = P_CONST * df["Hs"]**2 * df["Tp"]
    return df

def load_wis(path):
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    df.index.name = "datetime"
    df = df[["waveHs", "waveTp"]].rename(columns={"waveHs": "Hs", "waveTp": "Tp"})
    df = df.apply(pd.to_numeric, errors="coerce")
    df["P_kW_m"] = P_CONST * df["Hs"]**2 * df["Tp"]
    return df

ndbc = load_ndbc(NDBC_FILE)
print(f"NDBC 44020 loaded: {len(ndbc)} rows, {ndbc.index.min()} → {ndbc.index.max()}")

if WIS_FILE.exists():
    wis = load_wis(WIS_FILE)
    print(f"WIS ST63064 loaded: {len(wis)} rows, {wis.index.min()} → {wis.index.max()}")
else:
    wis = None
    print("WIS file not found — plotting NDBC only.")

ndbc.head()

In [ ]:
# ── Shared plot style ─────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":       "DejaVu Sans",
    "font.size":         10,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         True,
    "grid.color":        "#e0e0e0",
    "grid.linewidth":    0.6,
    "grid.linestyle":    "--",
    "axes.labelsize":    11,
    "axes.titlesize":    12,
    "axes.titleweight":  "bold",
    "legend.frameon":    False,
    "legend.fontsize":   9,
    "figure.dpi":        150,
})

date_fmt = mdates.DateFormatter("%b '%y")
date_loc = mdates.MonthLocator()

def style_xaxis(ax):
    ax.xaxis.set_major_locator(date_loc)
    ax.xaxis.set_major_formatter(date_fmt)
    ax.tick_params(axis="x", rotation=0)

def rolling(s, days=7):
    return s.rolling(f"{days}D", center=True, min_periods=24).mean()

## 2. Figure 0 — Station Location Map

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# ── Station coordinates ───────────────────────────────────────────────────────
stations = {
    "Marconi Beach\n(study site)": {"lon": -69.9741, "lat": 41.8977, "marker": "*", "color": "red",          "size": 250},
    "WIS ST63064":                 {"lon": -69.875,  "lat": 41.825,  "marker": "s", "color": "#E05C2A",      "size": 80},
    "NDBC 44020":                  {"lon": -70.279,  "lat": 41.493,  "marker": "^", "color": "#2176AE",      "size": 80},
}

fig4 = plt.figure(figsize=(9, 7))
ax   = fig4.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())

# Extent: zoom in on Cape Cod area
ax.set_extent([-71.5, -69.0, 41.0, 42.5], crs=ccrs.PlateCarree())

# Basemap features
ax.add_feature(cfeature.LAND,            facecolor="#f0ede8", zorder=0)
ax.add_feature(cfeature.OCEAN,           facecolor="#cce5f0", zorder=0)
ax.add_feature(cfeature.LAKES,           facecolor="#cce5f0", zorder=0)
ax.add_feature(cfeature.STATES,          edgecolor="#aaaaaa", linewidth=0.8, zorder=1)
ax.add_feature(cfeature.COASTLINE,       edgecolor="#555555", linewidth=0.8, zorder=2)
ax.add_feature(cfeature.BORDERS,         edgecolor="#888888", linewidth=0.5, zorder=1)
ax.add_feature(cfeature.NaturalEarthFeature(
    "physical", "rivers_lake_centerlines", "10m"),
    edgecolor="#aaccdd", facecolor="none", linewidth=0.5, zorder=1)

# Gridlines
gl = ax.gridlines(draw_labels=True, linewidth=0.4, color="grey",
                  alpha=0.5, linestyle="--")
gl.top_labels   = False
gl.right_labels = False

# Plot stations
for name, info in stations.items():
    ax.scatter(info["lon"], info["lat"],
               marker=info["marker"], color=info["color"], s=info["size"],
               transform=ccrs.PlateCarree(), zorder=5, edgecolors="k", linewidths=0.5)
    ax.text(info["lon"] + 0.05, info["lat"] + 0.04, name,
            transform=ccrs.PlateCarree(), fontsize=8.5, zorder=6,
            bbox=dict(facecolor="white", alpha=0.6, edgecolor="none", pad=1.5))

ax.set_title("Station Locations — Marconi Beach, Cape Cod", fontsize=12, fontweight="bold")

fig4.tight_layout()
fig4.savefig(PLOT_DIR / "station_map.png", bbox_inches="tight", dpi=150)
plt.show()
print("✓ Saved station_map.png")

## 2. Figure 1 — Time Series (Hs, Tp, Wave Power)

NDBC 44020 covers the full period (blue). WIS ST63064 overlaid for Oct–Dec 2024 (orange) where available.

In [ ]:
fig = plt.figure(figsize=(13, 10))
fig.suptitle(
    "Wave Conditions — Marconi Beach, Cape Cod\n"
    "NDBC 44020 (observed) & WIS ST63064 (hindcast)  |  Oct 2024 – Mar 2025",
    fontsize=13, fontweight="bold", y=0.98
)
gs = GridSpec(3, 1, figure=fig, hspace=0.45)

for i, (col, ylabel, title) in enumerate(zip(
    ["Hs", "Tp", "P_kW_m"],
    ["Hs  (m)", "Tp  (s)", "P  (kW m⁻¹)"],
    ["Significant Wave Height", "Peak Wave Period", "Wave Power  [ P = ρg²Hs²Tp / 64π ]"],
)):
    ax = fig.add_subplot(gs[i])

    # NDBC — full period
    ax.fill_between(ndbc.index, ndbc[col], alpha=0.15, color=PALETTE["ndbc"])
    ax.plot(ndbc.index, ndbc[col],
            color=PALETTE["ndbc"], lw=0.6, alpha=0.5, label="NDBC 44020 hourly")
    ax.plot(ndbc.index, rolling(ndbc[col]),
            color=PALETTE["ndbc"], lw=2.2, label="NDBC 7-day mean")

    # WIS — Oct–Dec 2024 overlay
    if wis is not None:
        ax.plot(wis.index, wis[col],
                color=PALETTE["wis"], lw=0.6, alpha=0.5, label="WIS ST63064 hourly")
        ax.plot(wis.index, rolling(wis[col]),
                color=PALETTE["wis"], lw=2.2, label="WIS 7-day mean")

    # Storm threshold on Hs panel only
    if col == "Hs":
        ax.axhline(3.0, color="#888", lw=1, ls=":", label="Storm threshold (3 m)")
        idx_max = ndbc[col].idxmax()
        ax.annotate(f"  {ndbc[col].max():.1f} m",
                    xy=(idx_max, ndbc[col].max()),
                    xytext=(5, 4), textcoords="offset points",
                    fontsize=8.5, color=PALETTE["ndbc"])

    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(loc="upper left", ncol=4)
    ax.set_ylim(bottom=0)
    ax.set_xlim(pd.Timestamp(START), pd.Timestamp(END))
    style_xaxis(ax)

fig.savefig(PLOT_DIR / "wave_timeseries.png", bbox_inches="tight")
plt.show()
print("✓ Saved wave_timeseries.png")

## 3. Figure 2 — Monthly Box Plots

In [ ]:
ndbc["month"] = ndbc.index.to_period("M")
months        = sorted(ndbc["month"].unique())
month_labels  = [str(m) for m in months]

def monthly_data(df, col):
    return [df[col][df["month"] == m].dropna().values for m in months]

bp_kwargs = dict(
    patch_artist=True,
    medianprops=dict(color="white", linewidth=2),
    boxprops=dict(facecolor=PALETTE["ndbc"], alpha=0.75),
    whiskerprops=dict(color="#555"),
    capprops=dict(color="#555"),
    flierprops=dict(marker="o", markersize=2, alpha=0.3,
                    markerfacecolor=PALETTE["ndbc"], markeredgecolor="none"),
)

fig2, axes = plt.subplots(1, 3, figsize=(14, 5))
fig2.suptitle(
    "Monthly Wave Statistics — NDBC 44020  |  Oct 2024 – Mar 2025",
    fontsize=13, fontweight="bold"
)

for ax, col, ylabel, title in zip(
    axes,
    ["Hs", "Tp", "P_kW_m"],
    ["Hs  (m)", "Tp  (s)", "P  (kW m⁻¹)"],
    ["Significant Wave Height", "Peak Wave Period", "Wave Power"],
):
    ax.boxplot(monthly_data(ndbc, col), labels=month_labels, **bp_kwargs)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight="bold")
    ax.tick_params(axis="x", rotation=30)
    ax.set_ylim(bottom=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", color="#e0e0e0", linewidth=0.6, linestyle="--")

fig2.tight_layout()
fig2.savefig(PLOT_DIR / "wave_monthly_boxplots.png", bbox_inches="tight")
plt.show()
print("✓ Saved wave_monthly_boxplots.png")

## 4. Figure 3 — Hs–Tp Scatter Coloured by Wave Power

In [ ]:
fig3, ax = plt.subplots(figsize=(7, 6))

sc = ax.scatter(
    ndbc["Tp"], ndbc["Hs"],
    c=ndbc["P_kW_m"], cmap="plasma",
    s=6, alpha=0.55, linewidths=0, rasterized=True
)
cbar = fig3.colorbar(sc, ax=ax, pad=0.02)
cbar.set_label("Wave power  (kW m⁻¹)", fontsize=10)

# Iso-power contours
tp_grid = np.linspace(2, 22, 200)
for P_iso, ls in [(10, ":"), (50, "--"), (100, "-"), (200, "-.")]:
    hs_iso = np.sqrt(P_iso / (P_CONST * tp_grid))
    mask   = (hs_iso >= 0) & (hs_iso <= 10)
    ax.plot(tp_grid[mask], hs_iso[mask], color="grey", lw=0.9, linestyle=ls, alpha=0.7)
    i = np.where(mask)[0]
    if len(i):
        ax.text(tp_grid[i[-1]] + 0.15, hs_iso[i[-1]], f"{P_iso} kW/m",
                fontsize=7.5, color="grey", alpha=0.9, va="center")

ax.set_xlabel("Peak period  Tp  (s)")
ax.set_ylabel("Significant wave height  Hs  (m)")
ax.set_title("Hs – Tp scatter  |  NDBC 44020\nColour = wave power, contours = iso-power lines",
             fontweight="bold")
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)

fig3.tight_layout()
fig3.savefig(PLOT_DIR / "wave_scatter_power.png", bbox_inches="tight", dpi=150)
plt.show()
print("✓ Saved wave_scatter_power.png")

## 5. Summary Statistics

In [ ]:
print("── NDBC 44020 ──")
for col, label in [("Hs", "Hs (m)"), ("Tp", "Tp (s)"), ("P_kW_m", "P (kW/m)")]:
    s = ndbc[col].dropna()
    print(f"  {label:14s}  mean={s.mean():.2f}  median={s.median():.2f}  "
          f"max={s.max():.2f}  p95={s.quantile(0.95):.2f}")

if wis is not None:
    print("\n── WIS ST63064 ──")
    for col, label in [("Hs", "Hs (m)"), ("Tp", "Tp (s)"), ("P_kW_m", "P (kW/m)")]:
        s = wis[col].dropna()
        print(f"  {label:14s}  mean={s.mean():.2f}  median={s.median():.2f}  "
              f"max={s.max():.2f}  p95={s.quantile(0.95):.2f}")